# Cal-DPO + LoRA (r=1) on UltraFeedback

Fine-tunes Llama-3.1-8B-Instruct on openbmb/UltraFeedback using Cal-DPO + LoRA (r=1).

**Runtime:** Requires A100 GPU (40GB+). Use Colab Pro+ or higher.

In [1]:
!pip install peft datasets transformers accelerate tqdm -q

In [ ]:
import os, math, random, gc
from dataclasses import dataclass
from typing import Union, List, Dict, Optional, Tuple, Iterable
import torch
import torch.nn.functional as F
from torch.nn.functional import log_softmax
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

# =============================================================================
# Config
# =============================================================================
HF_TOKEN = None   # ← paste your HF token here if using gated models

MODEL_NAME     = "meta-llama/Llama-3.2-3B-Instruct"
REF_MODEL_NAME = "meta-llama/Llama-3.2-3B"

# ---- UltraFeedback dataset ----
DATASET         = "openbmb/UltraFeedback"
DATASET_CONFIG  = None
SPLIT           = "train[:80%]"

# Train/Test split
TRAIN_FRAC      = 0.8
RNG_SEED        = 42
MIN_GAP         = 1.0  # require at least this helpfulness score gap

# Input limits
MAX_INPUT_TOKENS = 1024
MAX_GEN_TOKENS   = 512
SCORE_MAX_GEN_TOKENS = 256

DO_TUNE         = True
BATCH_SIZE = 1
TRAIN_BATCH_SIZE = 1

# Scoring
SCORING_MODE = "mean"  # "mean" or "sum"
LENGTH_PENALTY_ALPHA = 0.0

# Chat template
USE_CHAT_TEMPLATE = True

# Cal-DPO config
BETA_CALDPO = 0.5
USE_BETA_TUNING = True
BETAS = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]

# LoRA config
LORA_R = 1
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training config
LEARNING_RATE = 1e-4
BATCH_SIZE_TRAIN = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
WARMUP_STEPS = 100
MAX_GRAD_NORM = 1.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Device: {DEVICE} | Dtype: {DTYPE}")


In [ ]:
# =============================================================================
# Helpers
# =============================================================================
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

def _safe_str(x) -> Optional[str]:
    return x if isinstance(x, str) and x.strip() else None

def _get_int(value) -> Optional[int]:
    try:
        if value is None: return None
        if isinstance(value, bool): return int(value)
        if isinstance(value, int):  return value
        if isinstance(value, float): return int(round(value))
        s = str(value).strip()
        return int(round(float(s))) if s else None
    except Exception:
        return None

def binom_ci_95(pct: float, N: int) -> Tuple[float, float]:
    p = pct / 100.0
    se = math.sqrt(p * (1 - p) / max(N, 1))
    return max(0.0, 100*(p-1.96*se)), min(100.0, 100*(p+1.96*se))


In [ ]:
# =============================================================================
# UltraFeedback adapter
# =============================================================================
def _extract_text(c: Dict) -> Optional[str]:
    if not isinstance(c, dict):
        return None
    for k in ("text", "response", "output", "completion", "content"):
        v = c.get(k)
        if isinstance(v, str) and v.strip():
            return v
    res = c.get("result")
    if isinstance(res, dict):
        v = res.get("text") or res.get("response") or res.get("output")
        if isinstance(v, str) and v.strip():
            return v
    return None

def _to_float(x) -> Optional[float]:
    try:
        return float(x)
    except Exception:
        return None

def _extract_score(c: Dict) -> Optional[float]:
    """
    Extract a scalar score from completion annotations.
    Prefer helpfulness rating; fall back to any numeric fields.
    """
    if not isinstance(c, dict):
        return None
    ann = c.get("annotations")
    if isinstance(ann, dict):
        help_ = ann.get("helpfulness")
        if isinstance(help_, dict):
            r = help_.get("Rating") or help_.get("rating") or help_.get("score")
            r = _to_float(r)
            if r is not None:
                return r
        for key in ("overall", "quality", "correctness", "honesty", "safety"):
            sub = ann.get(key)
            if isinstance(sub, dict):
                r = sub.get("Rating") or sub.get("rating") or sub.get("score")
                r = _to_float(r)
                if r is not None:
                    return r
        # Any flat numeric
        for _, v in ann.items():
            fv = _to_float(v)
            if fv is not None:
                return fv
    for k in ("score", "rating", "rank"):
        fv = _to_float(c.get(k))
        if fv is not None:
            return fv
    return None

def adapt_ultrafeedback(record: Dict, min_gap: float = MIN_GAP) -> Optional[Dict]:
    """
    Build (prompt, chosen, rejected) by picking the highest-scored completion
    vs the lowest-scored completion, and drop near-ties (< min_gap).
    Returns prompt as a string (not message list).
    """
    prompt = record.get("instruction")
    if not isinstance(prompt, str) or not prompt.strip():
        return None

    comps = record.get("completions")
    if not isinstance(comps, list) or len(comps) < 2:
        return None

    pairs = []
    for c in comps:
        text = _extract_text(c)
        score = _extract_score(c)
        if isinstance(text, str) and text.strip() and score is not None:
            pairs.append((text, float(score)))

    if len(pairs) < 2:
        return None

    pairs.sort(key=lambda t: t[1])  # low ... high
    lo_txt, lo_s = pairs[0]
    hi_txt, hi_s = pairs[-1]
    if hi_s - lo_s < min_gap:
        return None

    return {"prompt": prompt, "chosen": hi_txt, "rejected": lo_txt}


In [ ]:
# =========================
# Prompt rendering
# =========================
def render_prompt_with_policy_template(
    policy_tokenizer,
    prompt_or_messages: Union[str, List[Dict[str,str]]]
) -> str:
    if USE_CHAT_TEMPLATE and hasattr(policy_tokenizer, "apply_chat_template"):
        try:
            if isinstance(prompt_or_messages, list):
                msgs = prompt_or_messages
            else:
                msgs = [{"role": "user", "content": str(prompt_or_messages).strip()}]
            return policy_tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            pass
    if isinstance(prompt_or_messages, list):
        p = "\n".join(f"{m.get('role','user').upper()}: {(m.get('content') or '').strip()}" for m in prompt_or_messages)
    else:
        p = str(prompt_or_messages).strip()
    return p if p.endswith("\n") else (p + "\n")

def render_prompt_list(policy_tokenizer, prompts: List[Union[str, List[Dict[str,str]]]]) -> List[str]:
    return [render_prompt_with_policy_template(policy_tokenizer, p) for p in prompts]

def concat_prompt_response_text(prompt_text: str, response: str) -> Tuple[str, str]:
    full_text = prompt_text + ("" if prompt_text.endswith("\n") else "\n") + str(response).strip()
    return full_text, prompt_text


In [ ]:
# =============================================================================
# Scoring  (eval path unchanged; training path added)
# =============================================================================
def _concat_prompt_response_text(prompt_text: str, response: str) -> Tuple[str, str]:
    full = prompt_text + response.strip()
    return full, prompt_text

@torch.no_grad()
def sequence_logprob_stats_text(
    model, tokenizer,
    prompt_texts: List[str],
    responses: List[str],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Returns (lp_sum [B], tok_count [B]).  tok_count=0 means fully truncated."""
    sums, counts = [], []
    for i in range(0, len(prompt_texts), BATCH_SIZE):
        p_batch = prompt_texts[i:i+BATCH_SIZE]
        r_batch = responses[i:i+BATCH_SIZE]
        batch_inputs, batch_prompt_lens = [], []
        for p_txt, r in zip(p_batch, r_batch):
            full_text, prompt_only = _concat_prompt_response_text(p_txt, r)
            toks_full = tokenizer(full_text, truncation=True,
                max_length=min(MAX_INPUT_TOKENS+MAX_GEN_TOKENS, tokenizer.model_max_length),
                return_tensors="pt", add_special_tokens=True)
            toks_prompt = tokenizer(prompt_only, truncation=True,
                max_length=MAX_INPUT_TOKENS, return_tensors="pt", add_special_tokens=True)
            batch_inputs.append(toks_full)
            batch_prompt_lens.append(toks_prompt["input_ids"].shape[-1])

        pad_id = tokenizer.pad_token_id or 0
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [bi["input_ids"].squeeze(0) for bi in batch_inputs], batch_first=True, padding_value=pad_id)
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [bi["attention_mask"].squeeze(0) for bi in batch_inputs], batch_first=True, padding_value=0)
        input_ids     = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits   = model(input_ids=input_ids, attention_mask=attention_mask).logits
        logprobs = log_softmax(logits.to(torch.float32), dim=-1)

        for b in range(input_ids.size(0)):
            p_len = batch_prompt_lens[b]
            L     = int(attention_mask[b].sum().item())
            if p_len >= L:
                sums.append(torch.tensor(0.0)); counts.append(torch.tensor(0)); continue
            end  = min(L, p_len + SCORE_MAX_GEN_TOKENS)
            targ = input_ids[b, p_len:end]
            pred = logprobs[b, p_len-1:end-1]
            lp_sum = pred.gather(-1, targ.unsqueeze(-1)).squeeze(-1).sum()
            sums.append(lp_sum.cpu())
            counts.append(torch.tensor(targ.numel()))
    return torch.stack(sums), torch.stack(counts)

def _score_from_stats(lp_sum: torch.Tensor, tok_count: torch.Tensor) -> torch.Tensor:
    if SCORING_MODE == "mean":
        return lp_sum.float() / torch.clamp(tok_count.float(), min=1.0)
    elif SCORING_MODE == "lp_alpha":
        return lp_sum.float() + LENGTH_PENALTY_ALPHA * tok_count.float()
    else:
        return lp_sum.float()

# ── ← NEW: gradient-retaining scorer for training ──────────────────────────
def get_logps_with_grad(
    model, tokenizer,
    prompt_texts: List[str],
    responses: List[str],
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Like sequence_logprob_stats_text but KEEPS the computation graph.
    Returns (scores [N], valid_mask [N]).  scores retains grad on DEVICE."""
    batch_scores, batch_valid = [], []
    max_len = MAX_INPUT_TOKENS + SCORE_MAX_GEN_TOKENS

    for p_txt, resp in zip(prompt_texts, responses):
        full_text, prompt_only = _concat_prompt_response_text(p_txt, resp)
        toks_full   = tokenizer(full_text,   truncation=True, max_length=max_len,
                                return_tensors="pt", add_special_tokens=True)
        toks_prompt = tokenizer(prompt_only, truncation=True, max_length=MAX_INPUT_TOKENS,
                                return_tensors="pt", add_special_tokens=True)
        input_ids     = toks_full["input_ids"].to(DEVICE)
        attention_mask = toks_full["attention_mask"].to(DEVICE)
        p_len         = toks_prompt["input_ids"].shape[-1]

        with torch.amp.autocast('cuda', dtype=DTYPE):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        logprobs = log_softmax(logits.float(), dim=-1)
        seq_len  = int(attention_mask.sum().item())

        if p_len >= seq_len:
            batch_scores.append(torch.tensor(0.0, device=DEVICE))
            batch_valid.append(False)
            continue

        end      = min(seq_len, p_len + SCORE_MAX_GEN_TOKENS)
        resp_ids = input_ids[0, p_len:end]
        resp_lps = logprobs[0, p_len-1:end-1]
        selected = resp_lps.gather(-1, resp_ids.unsqueeze(-1)).squeeze(-1)

        # Apply same scoring mode as eval path
        if SCORING_MODE == "mean":
            score = selected.mean()
        elif SCORING_MODE == "lp_alpha":
            score = selected.sum() + LENGTH_PENALTY_ALPHA * selected.numel()
        else:
            score = selected.sum()

        batch_scores.append(score)   # retains grad
        batch_valid.append(True)

    return torch.stack(batch_scores), torch.tensor(batch_valid)


In [ ]:
# =============================================================================
# Model loaders
# =============================================================================
def load_causal_lm(model_id: str, token: Optional[str], dtype=DTYPE, device=DEVICE):
    """Frozen loader — used for reference model."""
    tok = AutoTokenizer.from_pretrained(model_id, token=token, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=token, torch_dtype=dtype).to(device)
    model.eval()
    return tok, model

def load_policy_with_lora(model_id: str, token: Optional[str]):
    """← NEW: load policy and attach LoRA adapters."""
    print(f"Loading policy tokenizer from {model_id} …")
    tok = AutoTokenizer.from_pretrained(model_id, token=token, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    print(f"Loading policy base model …")
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=token, torch_dtype=DTYPE, device_map="auto")

    print(f"Attaching LoRA (r={LORA_R}, alpha={LORA_ALPHA}) …")
    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        lora_dropout=LORA_DROPOUT, bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return tok, model


In [ ]:
# =============================================================================
# DPO eval helpers  (unchanged — used for β tuning on TRAIN)
# =============================================================================
@dataclass
class DPOTune:
    beta: float = 1.0

@torch.no_grad()
def cache_dpo_margins(
    records: List[Dict],
    policy_model, ref_model, policy_tok, ref_tok=None
) -> Tuple[torch.Tensor, int]:
    """
    Returns Δ [N_valid] where Δ_i = s_pos_i − s_neg_i,
    s = score(policy) − score(ref).
    """
    prompts_raw  = [r["prompt"]   for r in records]
    prompt_texts = render_prompt_list(policy_tok, prompts_raw)
    resps_ch     = [r["chosen"]   for r in records]
    resps_rj     = [r["rejected"] for r in records]

    pol_sum_ch, pol_cnt_ch = sequence_logprob_stats_text(policy_model, policy_tok, prompt_texts, resps_ch)
    pol_sum_rj, pol_cnt_rj = sequence_logprob_stats_text(policy_model, policy_tok, prompt_texts, resps_rj)
    pol_score_ch = _score_from_stats(pol_sum_ch, pol_cnt_ch)
    pol_score_rj = _score_from_stats(pol_sum_rj, pol_cnt_rj)

    if ref_model is None:
        ref_score_ch = torch.zeros_like(pol_score_ch); ref_cnt_ch = torch.ones_like(pol_cnt_ch)
        ref_score_rj = torch.zeros_like(pol_score_rj); ref_cnt_rj = torch.ones_like(pol_cnt_rj)
    else:
        ref_sum_ch, ref_cnt_ch = sequence_logprob_stats_text(ref_model, ref_tok, prompt_texts, resps_ch)
        ref_sum_rj, ref_cnt_rj = sequence_logprob_stats_text(ref_model, ref_tok, prompt_texts, resps_rj)
        ref_score_ch = _score_from_stats(ref_sum_ch, ref_cnt_ch)
        ref_score_rj = _score_from_stats(ref_sum_rj, ref_cnt_rj)

    valid_ch = (pol_cnt_ch > 0) & (ref_cnt_ch > 0)
    valid_rj = (pol_cnt_rj > 0) & (ref_cnt_rj > 0)
    idx = [i for i in range(len(records)) if valid_ch[i].item() and valid_rj[i].item()]
    if not idx:
        raise RuntimeError("No valid pairs. Increase token limits.")

    s_pos  = pol_score_ch[idx] - ref_score_ch[idx]
    s_neg  = pol_score_rj[idx] - ref_score_rj[idx]
    return (s_pos - s_neg).float(), len(idx)

def dpo_loss_and_wr(delta: torch.Tensor, beta: float) -> Tuple[float, float]:
    loss = F.softplus(-beta * delta).mean().item()
    wr   = (delta > 0).float().mean().item() * 100.0
    return loss, wr

def tune_beta_dpo(
    records, policy_model, ref_model, policy_tok, ref_tok,
    betas, split_name="TRAIN"
) -> Tuple[list, DPOTune]:
    delta, N = cache_dpo_margins(records, policy_model, ref_model, policy_tok, ref_tok)
    trials, best = [], (float("inf"), -1.0, None)
    for beta in betas:
        loss, wr = dpo_loss_and_wr(delta, beta)
        trials.append((beta, loss, wr, N))
        lo, hi = binom_ci_95(wr, N)
        print(f"[TUNE/{split_name}] beta={beta:.3g} | loss={loss:.4f} | WR={wr:.2f}% [{lo:.1f}, {hi:.1f}]")
        if (loss < best[0]) or (abs(loss-best[0]) < 1e-6 and wr > best[1]):
            best = (loss, wr, beta)
    print(f"\n[DPO Tuning ({split_name})] Best beta={best[2]:.3g} "
          f"(loss={best[0]:.4f}, WR={best[1]:.2f}%) on {N} pairs")
    return trials, DPOTune(beta=best[2])


In [ ]:
# =============================================================================
# Cal-DPO loss  (← NEW)
# =============================================================================
def compute_caldpo_loss(
    pol_ch: torch.Tensor,   # policy scores, chosen   (with grad)
    pol_rj: torch.Tensor,   # policy scores, rejected (with grad)
    ref_ch: torch.Tensor,   # reference scores, chosen  (no grad, cached)
    ref_rj: torch.Tensor,   # reference scores, rejected
    beta: float,
) -> Tuple[torch.Tensor, Dict]:
    """
    Cal-DPO = DPO margin + calibration MSE
      r = pol − ref   (implicit reward)
      DPO  : softplus(-(r_pos − r_neg))
      Cal  : MSE(r_pos, +0.5/β) + MSE(r_neg, −0.5/β)
    """
    r_pos = pol_ch - ref_ch
    r_neg = pol_rj - ref_rj
    margin = r_pos - r_neg

    dpo_term = F.softplus(-margin).mean()

    t_pos = 0.5 / beta
    t_neg = -0.5 / beta
    cal_term = F.mse_loss(r_pos, torch.full_like(r_pos, t_pos)) + \
               F.mse_loss(r_neg, torch.full_like(r_neg, t_neg))

    total = dpo_term + cal_term

    metrics = {
        'loss_total':           total.item(),
        'loss_dpo':             dpo_term.item(),
        'loss_cal':             cal_term.item(),
        'margin_mean':          margin.mean().item(),
        'reward_chosen_mean':   r_pos.mean().item(),
        'reward_rejected_mean': r_neg.mean().item(),
    }
    return total, metrics


In [ ]:
# =============================================================================
# Cal-DPO training loop  (← NEW)
# =============================================================================
def train_one_epoch(policy_model, ref_model, tokenizer, train_records, beta: float):
    """
    Phase 1: cache all ref scores on GPU, then offload ref → CPU.
    Phase 2: LoRA training loop using cached ref scores (instant lookup).
    """
    policy_model.train()
    ref_model.eval()

    # ── Phase 1: cache ref ──────────────────────────────────────────────────
    print("Pre-caching reference scores …")
    prompt_texts_all = render_prompt_list(tokenizer, [r["prompt"] for r in train_records])
    chosen_all       = [r["chosen"]   for r in train_records]
    rejected_all     = [r["rejected"] for r in train_records]

    with torch.no_grad():
        ref_sum_ch, ref_cnt_ch = sequence_logprob_stats_text(ref_model, tokenizer, prompt_texts_all, chosen_all)
        ref_sum_rj, ref_cnt_rj = sequence_logprob_stats_text(ref_model, tokenizer, prompt_texts_all, rejected_all)
        ref_score_ch = _score_from_stats(ref_sum_ch, ref_cnt_ch)
        ref_score_rj = _score_from_stats(ref_sum_rj, ref_cnt_rj)
        ref_valid    = (ref_cnt_ch > 0) & (ref_cnt_rj > 0)

    # Store as plain Python floats — tiny memory footprint
    ref_cache = {}   # idx → (score_ch, score_rj, valid)
    for i in range(len(train_records)):
        ref_cache[i] = (
            ref_score_ch[i].item(),
            ref_score_rj[i].item(),
            ref_valid[i].item()
        )
    del ref_sum_ch, ref_cnt_ch, ref_sum_rj, ref_cnt_rj, ref_score_ch, ref_score_rj, ref_valid
    print(f"  Cached {len(ref_cache)} ref scores. Offloading ref to CPU …")
    ref_model.cpu()
    clear_memory()

    # ── Phase 2: training loop ──────────────────────────────────────────────
    indexed = list(enumerate(train_records))
    random.shuffle(indexed)

    optimizer  = torch.optim.AdamW(policy_model.parameters(), lr=LEARNING_RATE)
    total_steps = len(indexed) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS,
        num_training_steps=max(total_steps, 1)
    )

    print(f"\n{'='*80}")
    print(f"Cal-DPO LoRA Training — 1 epoch | batch={TRAIN_BATCH_SIZE} | "
          f"accum={GRAD_ACCUM_STEPS} | eff_batch={TRAIN_BATCH_SIZE*GRAD_ACCUM_STEPS} | "
          f"steps≈{total_steps} | beta={beta}")
    print(f"  Cal targets: r_pos → {0.5/beta:+.4f}, r_neg → {-0.5/beta:+.4f}")
    print(f"{'='*80}")

    total_metrics = {'loss_total':0, 'loss_dpo':0, 'loss_cal':0, 'margin_mean':0}
    num_batches   = 0
    pbar = tqdm(range(0, len(indexed), TRAIN_BATCH_SIZE), desc="Epoch 1")

    for batch_idx, start in enumerate(pbar):
        batch   = indexed[start:start+TRAIN_BATCH_SIZE]
        indices = [idx for idx, _   in batch]
        prompts = [rec["prompt"]   for _, rec in batch]
        chosen  = [rec["chosen"]   for _, rec in batch]
        rejected= [rec["rejected"] for _, rec in batch]

        prompt_texts = render_prompt_list(tokenizer, prompts)

        # Policy scores WITH grad
        pol_ch, valid_ch = get_logps_with_grad(policy_model, tokenizer, prompt_texts, chosen)
        pol_rj, valid_rj = get_logps_with_grad(policy_model, tokenizer, prompt_texts, rejected)

        # Ref scores from cache (no GPU transfer)
        ref_ch_vals, ref_rj_vals, valid_ref_list = [], [], []
        for idx in indices:
            sc, sr, v = ref_cache[idx]
            ref_ch_vals.append(sc)
            ref_rj_vals.append(sr)
            valid_ref_list.append(v)

        ref_ch    = torch.tensor(ref_ch_vals,    device=DEVICE)
        ref_rj    = torch.tensor(ref_rj_vals,    device=DEVICE)
        valid_ref = torch.tensor(valid_ref_list)

        valid = valid_ch & valid_rj & valid_ref
        if not valid.any():
            continue

        # Cal-DPO loss
        loss, metrics = compute_caldpo_loss(
            pol_ch[valid], pol_rj[valid],
            ref_ch[valid], ref_rj[valid],
            beta
        )
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        for k in total_metrics:
            total_metrics[k] += metrics.get(k, 0.0)
        num_batches += 1

        pbar.set_postfix({
            'loss': f"{metrics['loss_total']:.4f}",
            'dpo':  f"{metrics['loss_dpo']:.4f}",
            'cal':  f"{metrics['loss_cal']:.4f}",
            'lr':   f"{scheduler.get_last_lr()[0]:.2e}"
        })

    # Flush any leftover accumulation
    total_batches = len(range(0, len(indexed), TRAIN_BATCH_SIZE))
    if total_batches % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()

    avg = {k: v/max(num_batches,1) for k,v in total_metrics.items()}
    print(f"\n  Train avg — Loss: {avg['loss_total']:.4f} "
          f"(DPO: {avg['loss_dpo']:.4f}, Cal: {avg['loss_cal']:.4f}) | "
          f"Margin: {avg['margin_mean']:.4f}")
    return policy_model


In [ ]:
# =============================================================================
# ECE helpers
# =============================================================================
def compute_ece(margins: list, num_bins: int = 10) -> Tuple[float, List]:
    """
    Symmetric ECE@10 from pairwise margins.
    Each margin m yields two samples:
      (confidence=sigmoid(m),  label=1)   — original pair
      (confidence=sigmoid(-m), label=0)   — flipped pair
    This prevents the trivial all-ones label bias.
    Returns (ece, bin_details) where bin_details is a list of
      (bin_idx, avg_conf, avg_acc, gap, count).
    """
    import numpy as np
    margins_np = np.array(margins, dtype=np.float64)

    # Symmetric expansion
    confidences = []
    labels      = []
    for m in margins_np:
        # original: sigmoid(m), label=1
        confidences.append(1.0 / (1.0 + np.exp(-m)))
        labels.append(1)
        # flipped:  sigmoid(-m), label=0
        confidences.append(1.0 / (1.0 + np.exp(m)))
        labels.append(0)

    confidences = np.array(confidences)
    labels      = np.array(labels)

    # Map every sample to (predicted_label, confidence_in_that_prediction)
    pred_labels = (confidences >= 0.5).astype(int)
    # "confidence" = probability assigned to the predicted class
    conf = np.where(pred_labels == 1, confidences, 1.0 - confidences)

    ece  = 0.0
    details = []
    bin_edges = np.linspace(0.0, 1.0, num_bins + 1)
    total_n   = len(conf)

    for b in range(num_bins):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        if b == 0:
            in_bin = (conf >= lo) & (conf <= hi)
        else:
            in_bin = (conf > lo)  & (conf <= hi)
        n = int(in_bin.sum())
        if n == 0:
            continue
        acc_bin  = float((pred_labels[in_bin] == labels[in_bin]).mean())
        conf_bin = float(conf[in_bin].mean())
        gap      = abs(acc_bin - conf_bin)
        ece     += (n / total_n) * gap
        details.append((b + 1, conf_bin, acc_bin, gap, n))

    return float(ece), details

def print_ece_table(details: List, title: str = ""):
    """Pretty-print the ECE bin table."""
    if title:
        print(f"\n[{title}]")
    print("  Bin   Confidence   Accuracy     Gap          Count")
    print("  ----- ------------ ------------ ------------ --------")
    for (b, conf, acc, gap, n) in details:
        print(f"  {b:<5d} {conf:>12.4f} {acc:>12.4f} {gap:>12.4f} {n:>8d}")


# =============================================================================
# Evaluation  (Cal-DPO loss components + WR + ECE@10)
# =============================================================================
@torch.no_grad()
def evaluate_caldpo(policy_model, ref_model, tokenizer, records, beta: float) -> Dict:
    """Both models must be on GPU. Returns avg metrics + win_rate + ece10."""
    policy_model.eval()
    ref_model.eval()

    prompt_texts = render_prompt_list(tokenizer, [r["prompt"] for r in records])
    chosen       = [r["chosen"]   for r in records]
    rejected     = [r["rejected"] for r in records]

    # Score everything
    pol_sum_ch, pol_cnt_ch = sequence_logprob_stats_text(policy_model, tokenizer, prompt_texts, chosen)
    pol_sum_rj, pol_cnt_rj = sequence_logprob_stats_text(policy_model, tokenizer, prompt_texts, rejected)
    ref_sum_ch, ref_cnt_ch = sequence_logprob_stats_text(ref_model,    tokenizer, prompt_texts, chosen)
    ref_sum_rj, ref_cnt_rj = sequence_logprob_stats_text(ref_model,    tokenizer, prompt_texts, rejected)

    pol_sc_ch = _score_from_stats(pol_sum_ch, pol_cnt_ch)
    pol_sc_rj = _score_from_stats(pol_sum_rj, pol_cnt_rj)
    ref_sc_ch = _score_from_stats(ref_sum_ch, ref_cnt_ch)
    ref_sc_rj = _score_from_stats(ref_sum_rj, ref_cnt_rj)

    # Valid mask
    valid = (pol_cnt_ch > 0) & (pol_cnt_rj > 0) & (ref_cnt_ch > 0) & (ref_cnt_rj > 0)
    idx   = valid.nonzero(as_tuple=True)[0]
    if len(idx) == 0:
        raise RuntimeError("No valid pairs in eval set.")

    # Compute Cal-DPO on valid subset
    _, metrics = compute_caldpo_loss(
        pol_sc_ch[idx].to(DEVICE), pol_sc_rj[idx].to(DEVICE),
        ref_sc_ch[idx].to(DEVICE), ref_sc_rj[idx].to(DEVICE),
        beta
    )

    # Implicit rewards and margin
    r_pos  = pol_sc_ch[idx] - ref_sc_ch[idx]
    r_neg  = pol_sc_rj[idx] - ref_sc_rj[idx]
    margin = r_pos - r_neg                          # [N_valid]

    # Win rate
    wr = (margin > 0).float().mean().item() * 100.0

    # ECE@10 (symmetric) — margin is the raw score fed to sigmoid inside compute_ece
    margins_list = margin.cpu().tolist()
    ece10, ece_details = compute_ece(margins_list, num_bins=10)

    metrics['win_rate']    = wr
    metrics['n_pairs']     = len(idx)
    metrics['ece10']       = ece10
    metrics['ece_details'] = ece_details
    return metrics


In [ ]:
# =============================================================================
# Main — full pipeline
# =============================================================================

# ── 1. Load dataset ────────────────────────────────────────────────────────
print(f"Loading dataset: {DATASET} [{SPLIT}] config={DATASET_CONFIG} ...")
ds = load_dataset(DATASET, name=DATASET_CONFIG, split=SPLIT) if DATASET_CONFIG else load_dataset(DATASET, split=SPLIT)

print("Columns:", ds.column_names)
if len(ds) > 0:
    sample0 = ds[0]
    print("Row[0] keys:", list(sample0.keys()))

adapted: List[Dict] = []
for rec in ds:
    a = adapt_ultrafeedback(rec, min_gap=MIN_GAP)
    if a and a["prompt"] and a["chosen"] and a["rejected"]:
        adapted.append(a)

print(f"Adapted examples (after gap>={MIN_GAP}): {len(adapted)}")
if not adapted:
    raise RuntimeError(
        f"No valid examples after adapting UltraFeedback with MIN_GAP={MIN_GAP}. "
        "Try MIN_GAP=0.0 or expand SPLIT."
    )


# Sanity preview
ex = adapted[0]
print("Example:", {k: (str(ex[k])[:100]+"…") if len(str(ex[k]))>100 else str(ex[k])
                   for k in ("prompt","chosen","rejected")})



random.Random(RNG_SEED).shuffle(adapted)
n_train     = max(1, int(TRAIN_FRAC * len(adapted)))
train_records  = adapted[:n_train]
test_records   = adapted[n_train:]
print(f"Split: TRAIN={len(train_records)} | TEST={len(test_records)}")

# ── 2. Load models ─────────────────────────────────────────────────────────
print("\nLoading policy (+ LoRA) …")
tok_policy, policy = load_policy_with_lora(MODEL_NAME, HF_TOKEN)

print("\nLoading reference (frozen) …")
tok_ref, ref = load_causal_lm(REF_MODEL_NAME, HF_TOKEN)





In [ ]:


# ── 3. Tune β on TRAIN (DPO loss, eval-only — no grad needed) ──────────────
torch.set_grad_enabled(False)
if DO_TUNE:
    print("\nTuning β on TRAIN …")
    _, best_dpo = tune_beta_dpo(
        train_records, policy, ref, tok_policy, tok_ref,
        betas=BETAS, split_name="TRAIN"
    )
    BETA = best_dpo.beta
else:
    BETA = 1.0
print(f"Using beta = {BETA}")
torch.set_grad_enabled(True)

# ── 4. Pre-training TEST eval ──────────────────────────────────────────────
print(f"\n{'='*80}")
print("PRE-TRAINING TEST eval …")
print(f"{'='*80}")
torch.set_grad_enabled(False)
pre = evaluate_caldpo(policy, ref, tok_policy, test_records, BETA)
torch.set_grad_enabled(True)
lo, hi = binom_ci_95(pre['win_rate'], pre['n_pairs'])
print(f"  [PRE]  WR={pre['win_rate']:.2f}% [{lo:.1f},{hi:.1f}] | "
      f"ECE@10={pre['ece10']:.4f} | "
      f"Loss={pre['loss_total']:.4f} (DPO={pre['loss_dpo']:.4f} Cal={pre['loss_cal']:.4f}) | "
      f"N={pre['n_pairs']}")
print_ece_table(pre['ece_details'], title="ECE Calibration (TEST — pre-training)")


In [ ]:

# ── 5. LoRA Training (1 epoch) ─────────────────────────────────────────────
print(f"\n{'='*80}")
print("TRAINING …")
print(f"{'='*80}")

# Move ref to GPU for caching phase inside train_one_epoch
ref.cuda()
clear_memory()

policy = train_one_epoch(policy, ref, tok_policy, train_records, BETA)

# ── 6. Post-training TEST eval ─────────────────────────────────────────────
ref.cuda()
clear_memory()

print(f"\n{'='*80}")
print("POST-TRAINING TEST eval …")
print(f"{'='*80}")
torch.set_grad_enabled(False)
post = evaluate_caldpo(policy, ref, tok_policy, test_records, BETA)
torch.set_grad_enabled(True)
lo, hi = binom_ci_95(post['win_rate'], post['n_pairs'])
print(f"  [POST] WR={post['win_rate']:.2f}% [{lo:.1f},{hi:.1f}] | "
      f"ECE@10={post['ece10']:.4f} | "
      f"Loss={post['loss_total']:.4f} (DPO={post['loss_dpo']:.4f} Cal={post['loss_cal']:.4f}) | "
      f"N={post['n_pairs']}")
print_ece_table(post['ece_details'], title="ECE Calibration (TEST — post-training)")

# ── 7. Summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("SUMMARY (TEST set)")
print(f"{'='*80}")
print(f"  {'Metric':<28} {'Pre-train':>12} {'Post-train':>12} {'Delta':>12}")
print(f"  {'-'*28} {'-'*12} {'-'*12} {'-'*12}")
for label, key in [("Win Rate (%)",    "win_rate"),
                   ("ECE@10 (sym)",    "ece10"),
                   ("Loss Total",      "loss_total"),
                   ("Loss DPO",        "loss_dpo"),
                   ("Loss Cal",        "loss_cal"),
                   ("Reward Chosen",   "reward_chosen_mean"),
                   ("Reward Rejected", "reward_rejected_mean")]:
    p, q = pre.get(key, 0.0), post.get(key, 0.0)
    d    = q - p
    sign = "+" if d >= 0 else ""
    print(f"  {label:<28} {p:>12.4f} {q:>12.4f} {sign}{d:>11.4f}")
print(f"  {'beta':<28}   {BETA}")
print(f"  {'Cal targets':<28}   r_pos→{0.5/BETA:+.4f}  r_neg→{-0.5/BETA:+.4f}")
print(f"{'='*80}")


In [ ]:
from google.colab import runtime
runtime.unassign()  # disconnects + releases the VM
